# Write and Run an Operation

Create a native text transformation with exact input lineage. `Executor.run` accepts definitions authored in this notebook and keeps invocation files alive until executor exit. It does not publish a store or claim persistent cache reuse.

For a pipeline, put the definition in an installed application package first; see [Writing Creator Operations](../../how-to-guides/writing-creator-operations.md).

In [ ]:
from __future__ import annotations

from enum import StrEnum
from pathlib import Path
from typing import Any, ClassVar

from pydantic import BaseModel, Field

from artisan.execution import Executor
from artisan.operations.base import OperationDefinition, PerArtifact
from artisan.schemas import (
    ArtifactResult,
    BatchStrategy,
    CSVArtifact,
    ExecuteInput,
    InputSpec,
    OutputSpec,
    PostprocessInput,
    PreprocessInput,
)

Declare the accepted roles and the output's parent role. Every declared parent must have an exact reference; a bare content ID cannot identify repeated occurrences.

In [ ]:
class TextUppercaser(OperationDefinition):
    name = "tutorial_text_uppercaser"
    description = "Uppercase input text while preserving its exact source position"

    class InputRole(StrEnum):
        document = "document"

    class OutputRole(StrEnum):
        document = "document"

    inputs: ClassVar[dict[str, InputSpec]] = {
        "document": InputSpec(artifact_type="csv")
    }
    outputs: ClassVar[dict[str, OutputSpec]] = {
        "document": OutputSpec(
            artifact_type="csv", derives_from={"inputs": ["document"]}
        )
    }

    class Params(BaseModel):
        add_header: bool = Field(
            default=False, description="Prepend an uppercase marker"
        )

    params: Params = Params()

    def preprocess(self, inputs: PreprocessInput) -> dict[str, Any]:
        return {
            "documents": PerArtifact(
                [
                    {
                        "path": artifact.materialized_path,
                        "name": artifact.original_name,
                        "parent": inputs.ref("document", index=index),
                    }
                    for index, artifact in enumerate(inputs.input_artifacts["document"])
                ]
            )
        }

    def execute_function(self, inputs: ExecuteInput) -> dict[str, Any]:
        outputs = []
        for index, item in enumerate(inputs.inputs["documents"]):
            text = Path(item["path"]).read_text().upper()
            if self.params.add_header:
                text = "=== UPPERCASED ===\n" + text
            target = Path(inputs.execute_dir) / f"document_{index}.csv"
            target.write_text(text)
            outputs.append({**item, "path": target})
        return {"outputs": outputs}

    def postprocess(self, inputs: PostprocessInput) -> ArtifactResult:
        result = ArtifactResult()
        for item in inputs.memory_outputs["outputs"]:
            result.add_artifact(
                "document",
                CSVArtifact.draft(
                    content=item["path"].read_bytes(),
                    original_name=item["name"] + ".csv",
                ),
                sources={"document": [item["parent"]]},
            )
        return result

`PerArtifact` marks values the runtime slices for each native call. Carry the reference with its file/result rather than rebuilding parents from content equality. Drafts may omit a step origin; standalone outputs keep that origin unset.

In [ ]:
first = CSVArtifact.draft(b"label\nalpha\n", "first.csv").finalize()
second = CSVArtifact.draft(b"label\nbeta\n", "second.csv").finalize()
inputs = [first, second, first]
expected = [artifact.content.upper() for artifact in inputs]

for size in (1, 2):

    class BatchedUppercaser(TextUppercaser):
        name = f"tutorial_uppercaser_{size}"
        batch_strategy: BatchStrategy = BatchStrategy(artifacts_per_unit=size)

    with Executor() as executor:
        result = executor.run(BatchedUppercaser, inputs={"document": inputs})
        assert [
            artifact.content for artifact in result.artifacts["document"]
        ] == expected
        assert [edge.source_input_index for edge in result.lineage["document"]] == [
            0,
            1,
            2,
        ]
        assert all(
            artifact.is_finalized and artifact.origin_step_number is None
            for artifact in result.artifacts["document"]
        )
    assert executor.cleanup_complete
print("Both batch sizes preserved the repeated first input and all three exact parents")

Files created by an invocation belong to its executor. Copy needed external results before closing it. For durable outputs, move this definition into an installed package and use PipelineManager; do not send a notebook-local class to a fresh worker.